# Krishi Drishti - P11 & P12: Crop Recommendation & Yield Prediction
- **P11**: 5-fold cross-validated Random Forest & XGBoost on soil N-P-K and weather inputs (99.55% CV accuracy).
- **P12**: Historical Yield Prediction (1997-2020) with XGBoost regressor (R²=0.97) and disease penalty adjustments.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

crop_df = pd.read_csv("Dataset/crop_recommendation.csv")
print("Crop recommendation dataset shape:", crop_df.shape)
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
X = crop_df[features].values
y = crop_df["label"].values

rf = RandomForestClassifier(n_estimators=100, random_state=42)
skf = StratifiedKFold(5, shuffle=True, random_state=42)
accs = [accuracy_score(y[v], rf.fit(X[t], y[t]).predict(X[v])) for t, v in skf.split(X, y)]
print(f"5-Fold CV Accuracy: {np.mean(accs)*100:.2f}%")


### P12: Yield Prediction with Disease Loss Adjustment


In [ ]:
import joblib
yield_pipe = joblib.load("ml-training/models/yield_predictor.joblib")
with open("ml-training/models/yield_metadata.json") as f:
    import json; meta = json.load(f)

# Prediction test for farmer
test_sample = pd.DataFrame([{
    "Crop": "Wheat", "Season": "Rabi", "State": "Punjab",
    "Area": 5.0, "Annual_Rainfall": 650.0, "Fertilizer": 80000.0, "Pesticide": 400.0
}])
base_yield = yield_pipe.predict(test_sample)[0]
adjusted_yield = base_yield * meta["disease_penalty_factors"]["medium"]
print(f"Base Predicted Yield: {base_yield:.2f} tonnes/ha")
print(f"Medium Disease Adjusted Yield: {adjusted_yield:.2f} tonnes/ha (-28% loss)")
